In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
print(os.listdir("/kaggle/input/datasets/ashishpathak778/cifar10"))

In [ ]:
import os

base_path = "/kaggle/input/datasets/ashishpathak778/cifar10/cifar-10-batches-py"
print(os.listdir(base_path))




In [ ]:
import torchvision
import torchvision.transforms as transforms

transform = transforms.ToTensor()

train_dataset = torchvision.datasets.CIFAR10(
    root="/kaggle/input/datasets/ashishpathak778/cifar10",
    train=True,
    download=False,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="/kaggle/input/datasets/ashishpathak778/cifar10",
    train=False,
    download=False,
    transform=transform
)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

In [ ]:
import pickle
import numpy as np
import time
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import SGD, Adagrad, RMSprop, Adam, Adadelta
from tensorflow.keras.utils import to_categorical

# Your base path
base_path = "/kaggle/input/datasets/ashishpathak778/cifar10/cifar-10-batches-py"


def load_batch(file):
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict


# Load training data
X_train = []
y_train = []

for i in range(1, 6):
    batch = load_batch(f"{base_path}/data_batch_{i}")
    X_train.append(batch[b'data'])
    y_train += batch[b'labels']

X_train = np.concatenate(X_train)
y_train = np.array(y_train)

# Load test data
test_batch = load_batch(f"{base_path}/test_batch")
X_test = test_batch[b'data']
y_test = np.array(test_batch[b'labels'])

# Reshape
X_train = X_train.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
X_test = X_test.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)

# Normalize
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# One-hot encoding
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

In [ ]:
def build_mlp(optimizer):
    model = Sequential([
        Flatten(input_shape=(32, 32, 3)),
        Dense(512, activation='relu'),
        Dense(256, activation='relu'),
        Dense(10, activation='softmax')
    ])
    
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [ ]:
# results = {}

# def train_and_evaluate(name, optimizer, batch_size):
#     print(f"\nTraining with {name}...")
#     model = build_mlp(optimizer)
    
#     start = time.time()
#     model.fit(X_train, y_train, epochs=10, batch_size=batch_size, verbose=0)
#     end = time.time()
    
#     loss, acc = model.evaluate(X_test, y_test, verbose=0)
    
#     results[name] = {
#         "Accuracy": acc,
#         "Training Time (s)": end - start
#     }


# # Batch GD
# train_and_evaluate("Batch GD", SGD(learning_rate=0.01), batch_size=50000)

# # SGD
# train_and_evaluate("Stochastic GD", SGD(learning_rate=0.01), batch_size=1)

# # Mini-batch GD
# train_and_evaluate("Mini-batch GD", SGD(learning_rate=0.01), batch_size=32)

# # Momentum
# train_and_evaluate("Momentum GD", SGD(learning_rate=0.01, momentum=0.9), batch_size=32)

# # Nesterov
# train_and_evaluate("Nesterov GD", SGD(learning_rate=0.01, momentum=0.9, nesterov=True), batch_size=32)

# # Adagrad
# train_and_evaluate("Adagrad", Adagrad(learning_rate=0.01), batch_size=32)

# # RMSprop
# train_and_evaluate("RMSprop", RMSprop(learning_rate=0.001), batch_size=32)

# # Adadelta
# train_and_evaluate("Adadelta", Adadelta(), batch_size=32)

# # Adam
# train_and_evaluate("Adam", Adam(learning_rate=0.001), batch_size=32)

In [ ]:
# df_results = pd.DataFrame(results).T
# df_results

In [ ]:
def build_base_mlp():
    model = Sequential([
        Flatten(input_shape=(32, 32, 3)),
        Dense(512, activation='relu'),
        Dense(256, activation='relu'),
        Dense(10, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

****L2 Regularization****

In [ ]:
from tensorflow.keras.regularizers import l2

def build_l2_mlp():
    model = Sequential([
        Flatten(input_shape=(32, 32, 3)),
        Dense(512, activation='relu', kernel_regularizer=l2(0.001)),
        Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
        Dense(10, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

**Dropout**

In [ ]:
from tensorflow.keras.layers import Dropout

def build_dropout_mlp():
    model = Sequential([
        Flatten(input_shape=(32, 32, 3)),
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(10, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

**Dataset Augmentation**

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

datagen.fit(X_train)

In [ ]:
model_aug = build_base_mlp()
model_aug.fit(datagen.flow(X_train, y_train, batch_size=32),
              epochs=10,
              validation_data=(X_test, y_test))

**Adding Noise to Inputs**

In [ ]:
from tensorflow.keras.layers import GaussianNoise

def build_noise_mlp():
    model = Sequential([
        Flatten(input_shape=(32, 32, 3)),
        GaussianNoise(0.1),
        Dense(512, activation='relu'),
        Dense(256, activation='relu'),
        Dense(10, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

**Early Stopping**

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

model_es = build_base_mlp()
model_es.fit(X_train, y_train,
             epochs=50,
             batch_size=32,
             validation_split=0.2,
             callbacks=[early_stop])

**Ensemble Methods**

In [ ]:
models = []
for i in range(3):
    model = build_base_mlp()
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    models.append(model)

# Ensemble prediction
preds = np.mean([model.predict(X_test) for model in models], axis=0)
ensemble_acc = np.mean(np.argmax(preds, axis=1) == np.argmax(y_test, axis=1))
print("Ensemble Accuracy:", ensemble_acc)

Parameter Sharing and Tying (MLP Adaptation)

In [ ]:
shared_dense = Dense(256, activation='relu')

def build_shared_mlp():
    model = Sequential([
        Flatten(input_shape=(32, 32, 3)),
        shared_dense,
        shared_dense,
        Dense(10, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

**Create Comparision Function**

In [ ]:
results_reg = {}

def train_and_compare(name, model):
    print(f"\nTraining: {name}")
    
    history = model.fit(
        X_train, y_train,
        epochs=15,
        batch_size=32,
        validation_split=0.2,
        verbose=0
    )
    
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    
    results_reg[name] = {
        "Train Accuracy": history.history['accuracy'][-1],
        "Val Accuracy": history.history['val_accuracy'][-1],
        "Test Accuracy": test_acc
    }

**Base Model**

In [ ]:
train_and_compare("Base MLP", build_base_mlp())

**L2 Regularization**

In [ ]:
train_and_compare("L2 Regularization", build_l2_mlp())

**Dropout**

In [ ]:
train_and_compare("Dropout", build_dropout_mlp())

**Noise**

In [ ]:
train_and_compare("Input Noise", build_noise_mlp())

**Early Stopping**

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

model_es = build_base_mlp()

history = model_es.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)

test_loss, test_acc = model_es.evaluate(X_test, y_test, verbose=0)

results_reg["Early Stopping"] = {
    "Train Accuracy": history.history['accuracy'][-1],
    "Val Accuracy": history.history['val_accuracy'][-1],
    "Test Accuracy": test_acc
}

**Dataset Augmentation**

In [ ]:
model_aug = build_base_mlp()

history = model_aug.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=15,
    validation_data=(X_test, y_test),
    verbose=0
)

test_loss, test_acc = model_aug.evaluate(X_test, y_test, verbose=0)

results_reg["Data Augmentation"] = {
    "Train Accuracy": history.history['accuracy'][-1],
    "Val Accuracy": history.history['val_accuracy'][-1],
    "Test Accuracy": test_acc
}

**Ensemble**

In [ ]:
models = []
for i in range(3):
    model = build_base_mlp()
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    models.append(model)

preds = np.mean([m.predict(X_test) for m in models], axis=0)
ensemble_acc = np.mean(np.argmax(preds, axis=1) == np.argmax(y_test, axis=1))

results_reg["Ensemble"] = {
    "Train Accuracy": "—",
    "Val Accuracy": "—",
    "Test Accuracy": ensemble_acc
}

In [ ]:
import pandas as pd

df_reg = pd.DataFrame(results_reg).T
df_reg

In [ ]:
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()

In [ ]:
from tensorflow.keras.utils import to_categorical

# Normalize
X_train = X_train.astype('float32') / 255
X_test = X_test.astype('float32') / 255

# One-hot encoding
n_classes = 10
Y_train = to_categorical(y_train, n_classes)
Y_test = to_categorical(y_test, n_classes)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv2D, MaxPool2D, Flatten

model = Sequential()

# First Convolution Block
model.add(Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)))
model.add(MaxPool2D((2,2)))

# Second Convolution Block
model.add(Conv2D(64, (3,3), activation='relu', padding='same'))
model.add(MaxPool2D((2,2)))

# Flatten
model.add(Flatten())

# Fully Connected Layer
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

# Output Layer
model.add(Dense(10, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train, Y_train,
    batch_size=128,
    epochs=15,
    validation_data=(X_test, Y_test)
)

In [ ]:
test_loss, test_acc = model.evaluate(X_test, Y_test)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)